In [ ]:
import json
import pandas as pd

## Step 1: Load the raw dataset

The input file is `case.json`, stored inside the `datasets` folder.

We first load the file and inspect the number of records and the top-level structure.

In [ ]:
with open("datasets/case.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total records:", len(data))
print("Top-level keys:", data[0].keys())

## Step 2: Inspect sample records

The `Payload` column is stored as a JSON string, not as a ready Python object.

So we parse it using `json.loads()` to understand the structure of each event type.

In [ ]:
for row in data[:5]:
    payload_parsed = json.loads(row["Payload"])
    print("\nEventName:", row["EventName"])
    print("Parsed payload type:", type(payload_parsed))
    print("Parsed payload:", payload_parsed)

## Step 3: Count event types

Before transforming the data, it is useful to profile it and understand how many records belong to each category.

We found three relevant groups:
- `CurateOffer_Result`
- `DynamicPrice_Result` with provider `ApplyDynamicPriceRange`
- `DynamicPrice_Result` with provider `ApplyDynamicPricePerOption`

In [ ]:
curate_count = 0
range_count = 0
option_count = 0

for row in data:
    payload = json.loads(row["Payload"])
    
    if row["EventName"] == "CurateOffer_Result":
        curate_count += 1
    
    elif row["EventName"] == "DynamicPrice_Result":
        provider = payload["provider"]
        
        if provider == "ApplyDynamicPriceRange":
            range_count += 1
        elif provider == "ApplyDynamicPricePerOption":
            option_count += 1

print("CurateOffer_Result count:", curate_count)
print("DynamicPrice_Result with ApplyDynamicPriceRange count:", range_count)
print("DynamicPrice_Result with ApplyDynamicPricePerOption count:", option_count)

## Step 4: Inspect the `CurateOffer_Result` structure

For `CurateOffer_Result`:
- the payload is a list of offers
- each offer contains an `options` list
- each option must become one final row in the output table

This means we need to flatten nested lists.

In [ ]:
for row in data:
    if row["EventName"] == "CurateOffer_Result":
        payload = json.loads(row["Payload"])
        
        print("Type of payload:", type(payload))
        print("Number of offers in this payload:", len(payload))
        
        first_offer = payload[0]
        print("\nKeys in first offer:")
        print(first_offer.keys())
        
        print("\nType of options:", type(first_offer["options"]))
        print("Number of options in first offer:", len(first_offer["options"]))
        
        print("\nKeys in first option:")
        print(first_offer["options"][0].keys())
        
        break

## Step 5: Build `CuratedOfferOptions.csv`

Each row in the final table represents one option from the nested `options` list.

Fields come from three levels:
- outer row: `EnqueuedTimeUtc`
- offer level: `curationProvider`, `offerId`, `dealerId`
- option level: option-specific fields

In [ ]:
curated_rows = []

for row in data:
    if row["EventName"] == "CurateOffer_Result":
        payload = json.loads(row["Payload"])
        enqueued_time = row["EnqueuedTimeUtc"]
        
        for offer in payload:
            for option in offer["options"]:
                curated_rows.append({
                    "CurationProvider": offer.get("curationProvider"),
                    "OfferId": offer.get("offerId"),
                    "DealerId": offer.get("dealerId"),
                    "UniqueOptionId": option.get("uniqueOptionId"),
                    "OptionId": option.get("optionId"),
                    "IsMobileDealer": option.get("isMobileDealer"),
                    "IsOpen": option.get("isOpen"),
                    "Eta": option.get("eta"),
                    "ChamaScore": option.get("chamaScore"),
                    "ProductBrand": option.get("productBrand"),
                    "IsWinner": option.get("isWinner"),
                    "MinimumPrice": option.get("minimumPrice"),
                    "MaximumPrice": option.get("maximumPrice"),
                    "DynamicPrice": option.get("dynamicPrice"),
                    "FinalPrice": option.get("finalPrice"),
                    "DefeatPrimaryReason": option.get("defeatPrimaryReason"),
                    "DefeatReasons": option.get("defeatReasons"),
                    "EnqueuedTimeUtc": enqueued_time
                })

print("Total flattened curated rows:", len(curated_rows))
print(curated_rows[0])

## Step 6: Convert curated rows into a DataFrame

Now that the nested JSON has been flattened, we convert the rows into a tabular structure using Pandas.

In [ ]:
df_curated = pd.DataFrame(curated_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(df_curated.head())

## Step 7: Convert UTC timestamp to Brazil timezone (UTC-3)

The assignment requires the output date format as:

`DD/MM/YYYY`

We convert the original UTC timestamp to UTC-3 and then format it as a date string.

In [ ]:
df_curated["EnqueuedTimeUtc"] = pd.to_datetime(
    df_curated["EnqueuedTimeUtc"],
    format="%Y-%m-%d %H:%M:%S UTC"
)

df_curated["EnqueuedTimeSP"] = (
    df_curated["EnqueuedTimeUtc"] - pd.Timedelta(hours=3)
).dt.strftime("%d/%m/%Y")

print(df_curated[["EnqueuedTimeUtc", "EnqueuedTimeSP"]].head())

## Step 8: Export `CuratedOfferOptions.csv`

We keep only the required output columns and save the result as a CSV file.

In [ ]:
df_curated_final = df_curated[[
    "CurationProvider",
    "OfferId",
    "DealerId",
    "UniqueOptionId",
    "OptionId",
    "IsMobileDealer",
    "IsOpen",
    "Eta",
    "ChamaScore",
    "ProductBrand",
    "IsWinner",
    "MinimumPrice",
    "MaximumPrice",
    "DynamicPrice",
    "FinalPrice",
    "DefeatPrimaryReason",
    "DefeatReasons",
    "EnqueuedTimeSP"
]]

df_curated_final.to_csv("CuratedOfferOptions.csv", index=False)
print("CuratedOfferOptions.csv created!")

## Step 9: Inspect `DynamicPrice_Result` for per-option pricing

For provider `ApplyDynamicPricePerOption`:
- payload is a dictionary
- `algorithmOutput` is a list
- each item in that list becomes one output row

In [ ]:
for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPricePerOption":
            print("Payload keys:", payload.keys())
            print("Type of algorithmOutput:", type(payload["algorithmOutput"]))
            print("Length of algorithmOutput:", len(payload["algorithmOutput"]))
            print("First item in algorithmOutput:")
            print(payload["algorithmOutput"][0])
            break

## Step 10: Build `DynamicPriceOption.csv`

Each row in this output represents one item inside the `algorithmOutput` list for per-option pricing events.

In [ ]:
dynamic_option_rows = []

for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPricePerOption":
            enqueued_time = row["EnqueuedTimeUtc"]
            
            for item in payload["algorithmOutput"]:
                dynamic_option_rows.append({
                    "Provider": payload.get("provider"),
                    "OfferId": payload.get("offerId"),
                    "UniqueOptionId": item.get("uniqueOptionId"),
                    "BestPrice": item.get("bestPrice"),
                    "EnqueuedTimeUtc": enqueued_time
                })

print("Total dynamic option rows:", len(dynamic_option_rows))
print(dynamic_option_rows[0])

In [ ]:
df_option = pd.DataFrame(dynamic_option_rows)
print(df_option.head())

In [ ]:
df_option["EnqueuedTimeUtc"] = pd.to_datetime(
    df_option["EnqueuedTimeUtc"],
    format="%Y-%m-%d %H:%M:%S UTC"
)

df_option["EnqueuedTimeSP"] = (
    df_option["EnqueuedTimeUtc"] - pd.Timedelta(hours=3)
).dt.strftime("%d/%m/%Y")

In [ ]:
df_option_final = df_option[[
    "Provider",
    "OfferId",
    "UniqueOptionId",
    "BestPrice",
    "EnqueuedTimeSP"
]]

df_option_final.to_csv("DynamicPriceOption.csv", index=False)
print("DynamicPriceOption.csv created!")

## Step 11: Inspect `DynamicPrice_Result` for pricing range events

For provider `ApplyDynamicPriceRange`:
- payload is a dictionary
- `algorithmOutput` is also a dictionary
- each matching event becomes one output row

In [ ]:
for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPriceRange":
            print("Payload keys:", payload.keys())
            print("Type of algorithmOutput:", type(payload["algorithmOutput"]))
            print("algorithmOutput contents:", payload["algorithmOutput"])
            break

## Step 12: Build `DynamicPriceRange.csv`

Unlike the previous case, `algorithmOutput` is a dictionary here, so each event produces one final row.

In [ ]:
dynamic_range_rows = []

for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPriceRange":
            algo = payload["algorithmOutput"]
            enqueued_time = row["EnqueuedTimeUtc"]
            
            dynamic_range_rows.append({
                "Provider": payload.get("provider"),
                "OfferId": payload.get("offerId"),
                "MinGlobal": algo.get("min_global"),
                "MinRecommended": algo.get("min_recommended"),
                "MaxRecommended": algo.get("max_recommended"),
                "DifferenceMinRecommendMinTheory": algo.get("differenceMinRecommendMinTheory"),
                "EnqueuedTimeUtc": enqueued_time
            })

print("Total dynamic range rows:", len(dynamic_range_rows))
print(dynamic_range_rows[0])

In [ ]:
df_range = pd.DataFrame(dynamic_range_rows)
print(df_range.head())

In [ ]:
df_range["EnqueuedTimeUtc"] = pd.to_datetime(
    df_range["EnqueuedTimeUtc"],
    format="%Y-%m-%d %H:%M:%S UTC"
)

df_range["EnqueuedTimeSP"] = (
    df_range["EnqueuedTimeUtc"] - pd.Timedelta(hours=3)
).dt.strftime("%d/%m/%Y")

In [ ]:
df_range_final = df_range[[
    "Provider",
    "OfferId",
    "MinGlobal",
    "MinRecommended",
    "MaxRecommended",
    "DifferenceMinRecommendMinTheory",
    "EnqueuedTimeSP"
]]

df_range_final.to_csv("DynamicPriceRange.csv", index=False)
print("DynamicPriceRange.csv created!")